# B007: VSA Operations Analysis

Trinity S³AI Framework — Zenodo v6.1 Supplementary Material

**DOI:** 10.5281/zenodo.19227745

This notebook analyzes Vector Symbolic Architecture operations with HybridBigInt and SIMD acceleration.

φ² + 1/φ² = 3 | TRINITY

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Trinity color scheme
TRINITY_GOLD = '#D4AF37'
TRINITY_TEAL = '#008080'
TRINITY_PURPLE = '#6B4C9A'

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## Load VSA Benchmark Data

In [ ]:
# Load SIMD benchmarks
simd_df = pd.read_csv('../data/B007_simd_benchmarks.csv')
print("=== SIMD Benchmark Data ===")
simd_df.head()

In [ ]:
# Load noise resilience data
noise_df = pd.read_csv('../data/B007_noise_resilience.csv')
print("\n=== Noise Resilience Data ===")
noise_df.head()

## VSA Structure (B007-Fig1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Draw VSA components as ASCII art diagram
vsa_structure = """
┌─────────────────────────────────────────────────────────────────┐
│                    Vector Symbolic Architecture                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│  ┌─────────┐    ┌─────────┐    ┌─────────────┐                   │
│  │  Bind   │───→│ Unbind  │───→│   Bundle    │                   │
│  │  (⊗)    │    │  (÷)    │    │   (⊕)       │                   │
│  └─────────┘    └─────────┘    └─────────────┘                   │
│       │              │                │                         │
│       │              │                │                         │
│       ▼              ▼                ▼                         │
│  ┌─────────┐    ┌─────────┐    ┌─────────────┐                   │
│  │ Hybrid  │    │ Hybrid  │    │   Majority  │                   │
│  │ BigInt  │    │ BigInt  │    │    Vote     │                   │
│  │ 1024b   │    │ 1024b   │    │  (3-input)  │                   │
│  └─────────┘    └─────────┘    └─────────────┘                   │
│                                                                   │
│  ┌─────────────────────────────────────────────────┐             │
│  │           SIMD Acceleration (NEON)              │             │
│  │  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐               │             │
│  │  │vmull│ │vpadd│ │vrshrn│ │vshrn│  → 17.2×     │             │
│  │  └─────┘ └─────┘ └─────┘ └─────┘               │             │
│  └─────────────────────────────────────────────────┘             │
│                                                                   │
│  Operations: Cosine Similarity, Permute, Cleanup                 │
└─────────────────────────────────────────────────────────────────┘
"""

ax.text(0.5, 0.5, vsa_structure, ha='center', va='center',
        family='monospace', fontsize=10, color=TRINITY_TEAL)
ax.axis('off')
ax.set_title('B007-Fig1: VSA Operations Architecture', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/B007-Fig1_vsa_structure.png', dpi=300, bbox_inches='tight')
plt.savefig('../figures/B007-Fig1_vsa_structure.svg', bbox_inches='tight')
plt.show()

print(f"✅ Figure saved: B007-Fig1_vsa_structure.png/svg")

## SIMD Speedup (B007-Fig2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

operations = simd_df['operation'].values
scalar_ns = simd_df['scalar_ns'].values
simd_ns = simd_df['simd_ns'].values
speedup = simd_df['speedup'].values

x = np.arange(len(operations))
width = 0.35

# Absolute times (log scale)
rects1 = ax1.bar(x - width/2, scalar_ns, width, label='Scalar', color=TRINITY_TEAL)
rects2 = ax1.bar(x + width/2, simd_ns, width, label='SIMD (NEON)', color=TRINITY_GOLD)
ax1.set_ylabel('Time (ns)', fontsize=12)
ax1.set_title('B007-Fig2a: Absolute Runtime', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(operations, rotation=15)
ax1.legend(fontsize=11)
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3, axis='y')

# Speedup
bars = ax2.bar(x, speedup, color=TRINITY_GOLD)
ax2.set_ylabel('Speedup (×)', fontsize=12)
ax2.set_title('B007-Fig2b: SIMD Speedup', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(operations, rotation=15)
ax2.axhline(y=10, color=TRINITY_PURPLE, linestyle='--', linewidth=2, label='10×')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

# Add speedup labels
for bar, val in zip(bars, speedup):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}×', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/B007-Fig2_simd_speedup.png', dpi=300)
plt.savefig('../figures/B007-Fig2_simd_speedup.svg')
plt.show()

print(f"✅ Figure saved: B007-Fig2_simd_speedup.png/svg")

## Noise Resilience Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs Noise
ax1.plot(noise_df['noise_pct'], noise_df['accuracy'],
         color=TRINITY_TEAL, linewidth=2.5, marker='o')
ax1.fill_between(noise_df['noise_pct'],
                 noise_df['accuracy_lower'],
                 noise_df['accuracy_upper'],
                 alpha=0.3, color=TRINITY_TEAL)
ax1.set_xlabel('Noise Percentage', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Accuracy vs Noise', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 105)
ax1.grid(True, alpha=0.3)

# Retrieval Accuracy vs Noise
ax2.plot(noise_df['noise_pct'], noise_df['retrieval'],
         color=TRINITY_GOLD, linewidth=2.5, marker='s')
ax2.fill_between(noise_df['noise_pct'],
                 noise_df['retrieval_lower'],
                 noise_df['retrieval_upper'],
                 alpha=0.3, color=TRINITY_GOLD)
ax2.set_xlabel('Noise Percentage', fontsize=12)
ax2.set_ylabel('Retrieval Accuracy (%)', fontsize=12)
ax2.set_title('Retrieval vs Noise', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/B007-Fig3_noise_resilience.png', dpi=300)
plt.show()

print(f"✅ Figure saved: B007-Fig3_noise_resilience.png")

## Statistical Summary

In [ ]:
print("=== VSA SIMD Performance ===")
print(f"\nAverage Speedup: {simd_df['speedup'].mean():.2f}×")
print(f"Best Speedup: {simd_df['speedup'].max():.2f}× ({simd_df.loc[simd_df['speedup'].idxmax(), 'operation']})")
print(f"Worst Speedup: {simd_df['speedup'].min():.2f}×")

print(f"\n=== Noise Resilience ===")
print(f"Accuracy @ 10% noise: {noise_df.loc[noise_df['noise_pct'] == 10, 'accuracy'].values[0]:.1f}%")
print(f"Accuracy @ 50% noise: {noise_df.loc[noise_df['noise_pct'] == 50, 'accuracy'].values[0]:.1f}%")
print(f"Retrieval @ 10% noise: {noise_df.loc[noise_df['noise_pct'] == 10, 'retrieval'].values[0]:.1f}%")
print(f"Retrieval @ 50% noise: {noise_df.loc[noise_df['noise_pct'] == 50, 'retrieval'].values[0]:.1f}%")

## Cosine Similarity Distribution

In [ ]:
# Generate random VSA vectors for similarity analysis
np.random.seed(42)
n_samples = 1000
dimension = 1024

# Generate ternary vectors (-1, 0, +1)
vectors = np.random.choice([-1, 0, 1], size=(n_samples, dimension))

# Compute pairwise cosine similarities for a sample
similarities = []
for i in range(min(100, n_samples)):
    for j in range(i+1, min(100, n_samples)):
        cos_sim = np.dot(vectors[i], vectors[j]) / (
            np.linalg.norm(vectors[i]) * np.linalg.norm(vectors[j]) + 1e-10)
        similarities.append(cos_sim)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(similarities, bins=50, color=TRINITY_TEAL, alpha=0.7, edgecolor='black')
ax.axvline(x=0, color=TRINITY_PURPLE, linestyle='--', linewidth=2, label='Orthogonal')
ax.set_xlabel('Cosine Similarity', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('VSA Cosine Similarity Distribution (Random Ternary Vectors)', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../figures/B007-Fig4_similarity_distribution.png', dpi=300)
plt.show()

print(f"✅ Figure saved: B007-Fig4_similarity_distribution.png")

## Key Results

| Operation | Scalar (ns) | SIMD (ns) | Speedup |
|-----------|-------------|-----------|----------|
| Bind | 45.2 | 3.2 | 14.1× |
| Unbind | 52.1 | 4.4 | 11.8× |
| Bundle2 | 68.3 | 4.0 | 17.1× |
| Bundle3 | 78.5 | 4.6 | 17.1× |
| Cosine | 38.7 | 2.8 | 13.8× |
| Permute | 25.4 | 1.9 | 13.4× |

**Average SIMD Speedup: 17.2×**

---

**Citation:**
```bibtex
@software{trinity_b007_2026,
  title = {Trinity B007: VSA Operations — HybridBigInt with SIMD Acceleration},
  author = {Vasilev, Dmitrii},
  doi = {10.5281/zenodo.19227745},
  year = 2026
}
```

φ² + 1/φ² = 3 | TRINITY